In [1]:
# 12 Policy limits: do claims ever exceed what the policy can pay?
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims, match_claims
from src.features import build_features

claims = load_claims()
KEYS = ("limit", "sum_insured", "insured", "cover", "max", "cap", "indemn")

for line in LINES:
    p = load_policies(line, "train")
    cols = [c for c in p.columns if any(k in c.lower() for k in KEYS)]
    print(f"\n===== {line}: limit-like columns in the policy table: {cols or 'none'}")
    for c in cols:
        print(f"  {c}: {p[c].dtype}, examples {p[c].dropna().unique()[:8].tolist()}")

    m = match_claims(claims, p, line, cap=None)
    print(f"  claims: {len(m):,}; largest incurred: {m.incurred_to_date.nlargest(5).round(0).tolist()}")

    if line == "motor":
        top = m.nlargest(15, "incurred_to_date")[["peril", "status", "incurred_to_date", "loss_date"]]
        print("  15 largest motor claims:\n", top.to_string(index=False))
        vc = m[m.incurred_to_date > 100_000].incurred_to_date.round(-3).value_counts().head(5)
        print("  most common amounts above 100k (a pile-up at one value would suggest a limit):\n", vc.to_string())
    else:
        F = build_features(line, "train", policies=p)
        si_col = "sum_insured" if "sum_insured" in F else None
        if si_col is None:
            print("  no sum_insured found in features")
            continue
        mm = m.merge(F[["row_id", si_col]], on="row_id", how="left")
        mm["ratio"] = mm.incurred_to_date / mm[si_col]
        over = mm[mm.ratio > 1]
        print(f"  claims above sum insured: {len(over):,} of {len(mm):,} ({len(over) / len(mm):.2%})")
        if len(over):
            print("  by peril:\n", over.groupby("peril").agg(n=("ratio", "size"), max_ratio=("ratio", "max"),
                                                          amount_over=("incurred_to_date", "sum")).round(2).to_string())
            excess_amt = (over.incurred_to_date - over[si_col]).sum()
            print(f"  amount above sum insured: {excess_amt:,.0f} "
                  f"({excess_amt / mm.incurred_to_date.sum():.2%} of all incurred)")
        print("  ratio incurred / sum insured, upper percentiles:",
              mm.ratio.quantile([0.99, 0.999, 1.0]).round(3).to_dict())


===== motor: limit-like columns in the policy table: ['cover_type']
  cover_type: str, examples ['comprehensive', 'third_party_only', 'third_party_fire_theft']
  claims: 110,496; largest incurred: [11894277.0, 1118868.0, 602050.0, 524359.0, 476827.0]
  15 largest motor claims:
         peril status  incurred_to_date  loss_date
bodily_injury   open       11894276.93 2023-06-25
bodily_injury closed        1118867.58 2021-10-30
bodily_injury   open         602049.58 2024-01-02
bodily_injury closed         524358.77 2022-09-01
bodily_injury   open         476826.62 2023-04-20
bodily_injury closed         440710.50 2021-05-11
bodily_injury closed         427881.80 2022-11-10
bodily_injury closed         422601.70 2022-07-08
bodily_injury closed         417250.13 2022-10-02
bodily_injury   open         406450.38 2024-01-05
bodily_injury closed         401561.88 2023-04-19
bodily_injury closed         381329.91 2022-06-28
bodily_injury closed         373057.26 2021-11-09
bodily_injury closed

In [2]:
# 12b Cover consistency: claims on perils the policy does not cover
for line, rules in [("motor", [("cover_type", "third_party_only", ["own_damage", "theft", "glass_hail"]),
                               ("cover_type", "third_party_fire_theft", ["own_damage", "glass_hail"])]),
                    ("buildings", [("accidental_damage_cover", False, ["accidental_damage"])]),
                    ("contents", [("accidental_damage_cover", False, ["accidental_damage"])])]:
    p = load_policies(line, "train")
    m = match_claims(claims, p, line, cap=None)
    print(f"\n===== {line}: perils {sorted(m.peril.unique())}")
    for col, val, perils in rules:
        mm = m.merge(p[["row_id", col]], on="row_id", how="left")
        bad = mm[(mm[col] == val) & mm.peril.isin(perils)]
        share_pol = (p[col] == val).mean()
        print(f"  {col} = {val} ({share_pol:.1%} of policies): {len(bad):,} claims on {perils}; "
              f"net {bad.net.sum():,.0f} ({bad.net.sum() / m.net.sum():.2%} of line net)")
        if len(bad):
            print("   by peril and status:\n", bad.groupby(["peril", "status"]).size().to_string())


===== motor: perils ['bodily_injury', 'glass_hail', 'own_damage', 'theft', 'third_party_prop']
  cover_type = third_party_only (9.0% of policies): 1 claims on ['own_damage', 'theft', 'glass_hail']; net 1,995 (0.00% of line net)
   by peril and status:
 peril       status
own_damage  closed    1
  cover_type = third_party_fire_theft (9.1% of policies): 3 claims on ['own_damage', 'glass_hail']; net 1,501 (0.00% of line net)
   by peril and status:
 peril       status
own_damage  closed    2
            open      1

===== buildings: perils ['accidental_damage', 'escape_of_water', 'fire', 'flood', 'storm', 'subsidence']
  accidental_damage_cover = False (60.1% of policies): 502 claims on ['accidental_damage']; net 425,656 (0.18% of line net)
   by peril and status:
 peril              status
accidental_damage  closed    490
                   open       12

===== contents: perils ['accidental_damage', 'escape_of_water', 'flood', 'theft']
  accidental_damage_cover = False (55.0% of policie